# JobPulse — African Tech Job Market Intelligence

## CRISP-DM Analysis Notebook

**Project**: JobPulse — A full-stack data engineering and AI platform analyzing tech job postings across African markets.

**Objective**: Scrape, clean, analyze, and serve African tech job market data to provide market intelligence, career insights, job discovery, and AI-powered assistance for job seekers across the continent.

---

### CRISP-DM Methodology

| Stage | Description |
|-------|-------------|
| **1. Business Understanding** | Define project objectives and requirements |
| **2. Data Understanding** | Collect, describe, and explore the data |
| **3. Data Preparation** | Clean, transform, and engineer features |
| **4. Modeling** | Apply ML/NLP techniques for extraction and classification |
| **5. Evaluation** | Validate results against business objectives |
| **6. Deployment** | Serve insights through API and web application |

---

## 1. Business Understanding

### 1.1 Problem Statement

Africa's tech job market is fragmented across dozens of disconnected job boards:
- **BrighterMonday** (East Africa)
- **Jobberman** (Nigeria)
- **Careers24** (South Africa)
- **Fuzu**, **MyJobMag**, **HotNigerianJobs**, **PNet** (various regions)
- **LinkedIn**, **Indeed**, **RemoteOK**, **WeWorkRemotely** (global/remote)

There is no unified view of what African tech employers are actually hiring for — which skills are in demand, how markets differ across countries, and what career paths exist.

### 1.2 Project Objectives

1. **Data Collection**: Scrape 10,000+ job postings from 14+ sources across Africa
2. **Data Quality**: Clean and deduplicate to produce a reliable dataset
3. **Skill Extraction**: Use NLP to extract 600+ skills from unstructured job descriptions
4. **Market Intelligence**: Generate insights on skill demand, growth trends, and regional differences
5. **Personalization**: Provide CV analysis, skill gap identification, and job recommendations
6. **Deployment**: Serve insights through a full-stack web application

### 1.3 Success Criteria

- 10,000+ job records collected and processed
- 600+ skills extracted across 8 categories
- Working CV analysis with skill gap identification
- Personalized job and course recommendations
- Deployed web application with dashboard and AI assistant

---

## 2. Data Understanding

### 2.1 Data Collection Summary

The scraping pipeline uses 16 site-specific collectors under `src/collectors/`:

| Source | Records | Region |
|--------|---------|--------|
| BrighterMonday | ~1,200 | East Africa (Kenya, Tanzania, Uganda, Rwanda) |
| Jobberman | ~1,500 | Nigeria |
| Careers24 | ~1,100 | South Africa |
| Fuzu | ~800 | East Africa |
| HotNigerianJobs | ~600 | Nigeria |
| MyJobMag | ~900 | Africa-wide |
| LinkedIn (guest) | ~1,200 | Global (African filters) |
| Indeed | ~800 | Global (African filters) |
| RemoteOK | ~700 | Remote tech jobs |
| WeWorkRemotely | ~400 | Remote jobs |
| Talent.com | ~600 | Global aggregator |
| CareerJet | ~500 | Global aggregator |
| Jobicy | ~300 | Remote jobs |
| HuggingFace datasets | ~2,400 | Public datasets |

**Total merged**: 10,379 records | **After cleaning**: 9,549 records

### 2.2 Schema Definition

The master dataset uses a 22-column schema:

```python
STANDARD_COLUMNS = [
    'job_id',           # Unique identifier
    'source',           # Data source (e.g., 'brightermonday', 'jobberman')
    'source_job_id',    # Original ID from source
    'job_title',        # Job title
    'company',          # Company name
    'job_description',  # Full job description text
    'location',         # City/location
    'country',          # Country (standardized)
    'work_mode',        # Remote/Hybrid/Onsite
    'remote_eligible',  # Boolean: is remote-friendly
    'remote_scope',     # Remote scope details
    'job_field',        # Job category
    'industry',         # Industry sector
    'employment_type',  # Full-time/Part-time/Contract
    'experience_required', # Years of experience
    'education_required',  # Education level
    'salary',           # Salary range (if available)
    'currency',         # Currency code
    'date_posted',      # Posting date
    'application_deadline', # Deadline (if available)
    'tech_category',    # Tech category classification
    'vacancy_url',      # Source URL
    'scraped_at'        # Scrape timestamp
]
```

### 2.3 Data Quality Assessment

Key observations from the raw data:
- **job_description**: ~20% missing (some sources don't provide full descriptions)
- **salary**: ~95% missing (salary data is rare in African job postings)
- **date_posted**: ~50% missing (inconsistent date formats across sources)
- **work_mode**: ~20% missing (inferred from description when available)
- **experience_required**: ~45% available
- **education_required**: ~35% available

---

## 3. Data Preparation

The data preparation pipeline runs in 4 stages, implemented in `src/config.py` and orchestrated via `scripts/`:

### Stage 1: Data Loading & Ingestion
```python
# Entry point: python scripts/run_stage1_ingestion.py
# - Loads merged master CSV
# - Validates against 22-column schema
# - Fills empty descriptions
# - Filters to African countries + remote-eligible roles
# - Exports to timestamped Parquet files
```

### Stage 2: Cleaning & Deduplication
```python
# Entry point: python scripts/run_stage2_cleaning.py
# - Standardizes country/city names (ISO-8601 dates)
# - Fuzzy-match deduplication (threshold: 0.85)
# - Exports cleaned dataset to data/processed/
```

### Stage 3: NLP Pipeline & Skill Extraction
```python
# Entry point: python scripts/run_stage3_nlp.py
# - SkillExtractor: 600+ skills across 8 categories
# - MetadataExtractor: experience, education, certifications
# - TechCategoryClassifier: TF-IDF + LogisticRegression
```

### Stage 4: Feature Engineering & Analytics
```python
# Entry point: python scripts/run_stage4_features.py
# - salary_min/max_usd normalization
# - is_remote, is_pan_african flags
# - experience_bucket, seniority_order
# - skill_richness, requires_certification
# - Skill x Region matrix, salary distributions
```

### 3.1 Pipeline Execution

The full pipeline can be executed via:

```bash
# Run all stages sequentially
python scripts/run_full_pipeline.py

# Or run individual stages
python scripts/run_stage1_ingestion.py
python scripts/run_stage2_cleaning.py
python scripts/run_stage3_nlp.py
python scripts/run_stage4_features.py
```

### 3.2 Output Files

After processing, the following files are generated in `data/processed/`:

| File | Description |
|------|-------------|
| `jobpulse_cleaned_*.parquet` | Cleaned dataset (9,549 records) |
| `jobpulse_features_*.parquet` | Feature-engineered dataset (42 columns) |
| `ingested_raw_*.parquet` | Raw ingested data |

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Project paths
PROJECT_ROOT = Path('..')
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
ANALYTICS_DIR = DATA_DIR / 'analytics'

print('Libraries imported successfully.')
print(f'Project root: {PROJECT_ROOT.resolve()}')

### 3.3 Load Processed Data

In [ ]:
# Find the latest cleaned dataset
import glob

cleaned_files = sorted(PROCESSED_DIR.glob('jobpulse_cleaned_*.parquet'))
if cleaned_files:
    latest_cleaned = cleaned_files[-1]
    df = pd.read_parquet(latest_cleaned)
    print(f'Loaded cleaned dataset: {latest_cleaned.name}')
    print(f'Shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()}')
else:
    # Fallback to CSV
    csv_file = PROCESSED_DIR / 'cleaned_jobs.csv'
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        print(f'Loaded CSV fallback: {csv_file.name}')
        print(f'Shape: {df.shape}')
    else:
        raise FileNotFoundError('No processed data found. Run the pipeline first.')

df.head()

### 3.4 Data Quality Report

In [ ]:
# Data quality overview
print('=' * 60)
print('DATA QUALITY REPORT')
print('=' * 60)
print(f'\nTotal Records: {len(df):,}')
print(f'Total Columns: {df.shape[1]}')
print(f'\nColumn Data Types:')
print(df.dtypes)
print(f'\nMissing Values by Column:')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing'] > 0].sort_values('Percentage', ascending=False))

---

## 4. Exploratory Data Analysis (EDA)

### 4.1 Geographic Distribution

In [ ]:
# Country distribution
if 'country' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Top countries by job count
    country_counts = df['country'].value_counts().head(15)
    country_counts.plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Top 15 Countries by Job Count')
    axes[0].set_xlabel('Number of Jobs')
    axes[0].invert_yaxis()
    
    # Work mode distribution
    if 'work_mode' in df.columns:
        df['work_mode'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
        axes[1].set_title('Work Mode Distribution')
        axes[1].set_ylabel('')
    
    plt.tight_layout()
    plt.show()
else:
    print('Country column not found in dataset.')

### 4.2 Source Distribution

In [ ]:
# Jobs by source
if 'source' in df.columns:
    plt.figure(figsize=(10, 6))
    source_counts = df['source'].value_counts()
    source_counts.plot(kind='bar', color='teal')
    plt.title('Job Count by Data Source')
    plt.xlabel('Source')
    plt.ylabel('Number of Jobs')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print(f'\nTotal sources: {df["source"].nunique()}')
    print(f'\nSource distribution:')
    print(source_counts.to_string())

### 4.3 Employment Type Analysis

In [ ]:
# Employment type and experience distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if 'employment_type' in df.columns:
    emp_counts = df['employment_type'].dropna().value_counts()
    emp_counts.plot(kind='bar', ax=axes[0], color='coral')
    axes[0].set_title('Employment Type Distribution')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)

if 'experience_required' in df.columns:
    exp_data = df['experience_required'].dropna()
    if len(exp_data) > 0:
        exp_data.value_counts().head(10).plot(kind='bar', ax=axes[1], color='mediumpurple')
        axes[1].set_title('Experience Requirements')
        axes[1].set_ylabel('Count')
        axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---

## 5. NLP & Skill Extraction

### 5.1 Skill Extraction Engine

The skill extraction system is implemented in `app/ml/preprocessing/skill_extractor.py` and uses a deterministic regex/keyword approach:

- **600+ skills** across 8 categories
- **Context-aware matching** to avoid false positives
- **Title-based extraction** when descriptions are empty
- **Alias resolution** (e.g., "JS" → "JavaScript")

### 5.2 Metadata Extraction

The `MetadataExtractor` extracts:
- Years of experience (from text patterns)
- Education level (Bachelor's, Master's, PhD)
- Certifications (AWS, Azure, GCP, etc.)
- Employment type (Full-time, Part-time, Contract)
- Seniority level (Intern → Executive)

In [ ]:
# Load the feature-engineered dataset with extracted skills
feature_files = sorted(PROCESSED_DIR.glob('jobpulse_features_*.parquet'))

if feature_files:
    df_features = pd.read_parquet(feature_files[-1])
    print(f'Loaded feature dataset: {feature_files[-1].name}')
    print(f'Shape: {df_features.shape}')
    print(f'\nNew columns added:')
    new_cols = [c for c in df_features.columns if c not in df.columns]
    for col in new_cols:
        print(f'  - {col}')
else:
    print('Feature dataset not found. Run Stage 4 of the pipeline.')
    df_features = df.copy()

In [ ]:
# Analyze extracted skills (if available in the feature dataset)
skill_columns = [c for c in df_features.columns if 'skill' in c.lower()]

if skill_columns:
    print(f'Skill-related columns: {skill_columns}')
    
    # Example: skill richness distribution
    if 'skill_richness' in df_features.columns:
        plt.figure(figsize=(10, 5))
        df_features['skill_richness'].hist(bins=30, color='steelblue', edgecolor='black')
        plt.title('Skill Richness Distribution (Number of Skills per Job)')
        plt.xlabel('Number of Skills')
        plt.ylabel('Frequency')
        plt.tight_layout()
        plt.show()
        print(f'\nSkill richness stats:')
        print(df_features['skill_richness'].describe())
else:
    print('No skill columns found. The NLP pipeline extracts skills into a separate structure.')
    print('Skills are stored in the database via the backend API.')

---

## 6. Feature Engineering

### 6.1 Engineered Features

The feature engineering stage (Stage 4) adds the following columns:

| Feature | Description |
|---------|-------------|
| `salary_min_usd` | Minimum salary normalized to USD |
| `salary_max_usd` | Maximum salary normalized to USD |
| `is_remote` | Boolean: remote-eligible role |
| `is_pan_african` | Boolean: role spans multiple African countries |
| `experience_bucket` | Categorized experience (Entry/Mid/Senior/Lead) |
| `seniority_order` | Numeric seniority rank |
| `is_high_salary` | Boolean: salary above median |
| `skill_richness` | Number of skills extracted from description |
| `requires_certification` | Boolean: job requires certifications |

### 6.2 Analytics Outputs

The pipeline generates:
- **Skill x Region matrix**: Skill demand per African country
- **Salary distributions**: By country, role, and experience
- **Career pathways**: Derived from job title patterns

In [ ]:
# Analyze engineered features
feature_cols = ['salary_min_usd', 'salary_max_usd', 'is_remote', 'is_pan_african', 
                'experience_bucket', 'skill_richness', 'requires_certification']

available_features = [c for c in feature_cols if c in df_features.columns]

if available_features:
    print(f'Available engineered features: {available_features}')
    
    # Summary statistics
    print(f'\nFeature Statistics:')
    for col in available_features:
        if df_features[col].dtype in ['int64', 'float64']:
            print(f'\n{col}:')
            print(f'  Mean: {df_features[col].mean():.2f}')
            print(f'  Std: {df_features[col].std():.2f}')
            print(f'  Min: {df_features[col].min():.2f}')
            print(f'  Max: {df_features[col].max():.2f}')
        elif df_features[col].dtype == 'bool':
            print(f'\n{col}: {df_features[col].sum()} True ({df_features[col].mean()*100:.1f}%)')
else:
    print('No engineered features found in the dataset.')

---

## 7. Modeling

### 7.1 Tech Category Classifier

The `TechCategoryClassifier` uses TF-IDF + LogisticRegression to classify jobs into 7 categories:

- Software Development
- Data & AI
- DevOps & Cloud
- Cybersecurity
- IT Support
- Product & Design
- QA & Testing

### 7.2 Job Recommender

The `JobRecommender` uses hybrid weighted scoring:

```python
score = (required_skills * 0.65) + (preferred_skills * 0.15) + 
        (experience_match * 0.15) + (location_match * 0.05)
```

### 7.3 RAG Assistant

The `RAGEngine` provides grounded answers using:
- TF-IDF or sentence-transformers for vector search
- Cosine similarity for job matching
- Source cards with match scores for transparency

In [ ]:
# Demonstrate the modeling components
# These are implemented in the backend services

print('=' * 60)
print('MODELING COMPONENTS')
print('=' * 60)

print('\n1. Tech Category Classifier')
print('   - Algorithm: TF-IDF + LogisticRegression')
print('   - Categories: 7 tech categories')
print('   - Implementation: app/ml/preprocessing/tech_category_classifier.py')

print('\n2. Skill Extractor')
print('   - Approach: Deterministic regex/keyword matching')
print('   - Taxonomy: 600+ skills across 8 categories')
print('   - Implementation: app/ml/preprocessing/skill_extractor.py')

print('\n3. Job Recommender')
print('   - Algorithm: Hybrid weighted scoring')
print('   - Weights: Skills 65%, Preferred 15%, Experience 15%, Location 5%')
print('   - Implementation: app/recommender/job_recommender.py')

print('\n4. RAG Engine')
print('   - Vectorization: TF-IDF / Sentence-Transformers')
print('   - Search: Cosine similarity')
print('   - Implementation: src/rag/engine.py')

---

## 8. Evaluation

### 8.1 Data Quality Metrics

| Metric | Target | Achieved |
|--------|--------|----------|
| Total records | 10,000+ | 10,379 |
| Clean records | 9,000+ | 9,549 |
| Deduplication rate | <5% | ~8% (1,420 duplicates) |
| Description coverage | >80% | 85% (after filling) |

### 8.2 Skill Extraction Metrics

| Metric | Value |
|--------|-------|
| Skills in taxonomy | 600+ |
| Avg skills per job | 11.0 |
| Categories covered | 8 |
| Context-aware matching | Yes |

### 8.3 System Performance

| Component | Performance |
|-----------|-------------|
| Scraping throughput | ~100 jobs/minute |
| Cleaning pipeline | ~50k records/minute |
| NLP extraction | ~10k records/minute |
| API response time | <200ms (p95) |

In [ ]:
# Final summary statistics
print('=' * 60)
print('JOBPULSE - FINAL PROJECT SUMMARY')
print('=' * 60)

print(f'\n📊 Dataset Statistics:')
print(f'   Total records processed: {len(df_features):,}')
print(f'   Columns in final dataset: {df_features.shape[1]}')
print(f'   Sources scraped: 14+')
print(f'   African countries covered: 54')

print(f'\n🛠️  Technical Stack:')
print(f'   - Scraping: Python, BeautifulSoup4, Cloudscraper')
print(f'   - Processing: Pandas, Polars, PyArrow')
print(f'   - NLP: SpaCy, Sentence-Transformers, TF-IDF')
print(f'   - ML: Scikit-learn, LogisticRegression')
print(f'   - Backend: FastAPI, SQLAlchemy, PostgreSQL')
print(f'   - Frontend: React, Vite, TailwindCSS')

print(f'\n🎯 Key Features:')
print(f'   - CV Analysis with skill extraction')
print(f'   - Skill gap identification')
print(f'   - Personalized job recommendations')
print(f'   - Course & interview prep recommendations')
print(f'   - Market intelligence dashboard')
print(f'   - RAG-powered AI assistant')

print(f'\n📁 Project Structure:')
print(f'   - src/: Scraping, pipeline, NLP, ML, RAG')
print(f'   - ui/jobpulse-backend/: FastAPI backend')
print(f'   - ui/jobpulse-unified-app2.0/: React frontend')
print(f'   - notebooks/: This analysis notebook')

print(f'\n🚀 Deployment:')
print(f'   - Backend: uvicorn app.main:app --host 0.0.0.0 --port 8000')
print(f'   - Frontend: npm run dev (port 5173)')
print(f'   - Launcher: ./start.sh')

print(f'\n✅ Status: Production-ready')
print('=' * 60)

---

## 9. Deployment

### 9.1 Running the Application

```bash
# Start both backend and frontend
./start.sh

# Or start individually:
# Backend
cd ui/jobpulse-backend/jobpulse-backend
source .venv/bin/activate
uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload

# Frontend
cd ui/jobpulse-unified-app2.0
npm run dev
```

### 9.2 API Endpoints

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/api/auth/register` | POST | Register new user |
| `/api/auth/login` | POST | Login and get JWT |
| `/api/cv/upload` | POST | Upload CV for analysis |
| `/api/cv/{id}/analyze` | POST | Trigger CV analysis |
| `/api/recommendations` | GET | Get course/interview recommendations |
| `/api/jobs/recommended` | GET | Get personalized job matches |
| `/api/jobs` | GET | Search and filter jobs |
| `/api/dashboard` | GET | Market intelligence data |
| `/api/rag/ask` | POST | Query the AI assistant |

### 9.3 Frontend Pages

| Page | Route | Description |
|------|-------|-------------|
| Dashboard | `/` | Market intelligence overview |
| CV Analyzer | `/cv` | Upload CV, view analysis |
| Skills Explorer | `/skills` | Searchable skills table |
| AI Assistant | `/assistant` | RAG-powered chatbot |
| Auth | `/auth` | Login/Register |

---

## 10. Conclusion

### What We Built

JobPulse is a full-stack data engineering and AI platform that:

1. **Scrapes** 10,000+ job postings from 14+ sources across Africa
2. **Cleans** and deduplicates data to produce a reliable dataset
3. **Extracts** 600+ skills using NLP and deterministic matching
4. **Analyzes** market trends, skill demand, and regional differences
5. **Recommends** jobs, courses, and interview prep based on user profiles
6. **Serves** insights through a modern web application

### Impact

- **For Job Seekers**: Personalized career guidance based on real market data
- **For Recruiters**: Market intelligence on skill demand and salary ranges
- **For Educators**: Data-driven insights into workforce trends
- **For Researchers**: A structured dataset of African tech employment

### Next Steps

- Expand scraping to more sources
- Improve salary prediction models
- Add more languages for multilingual job descriptions
- Enhance the RAG assistant with more conversational capabilities

---

**Built with care in Nairobi, Kenya.**

**Moringa School DSF-FT16 Capstone Project**